# 07 · Rango sugerido de precios (regresión cuantílica)

El tablero muestra, además del precio de referencia, un **rango sugerido**. Se obtiene entrenando dos modelos LightGBM con pérdida cuantílica: uno estima el percentil inferior del precio y otro el superior para un alojamiento con esas características.

Se evalúa con septiembre, solo en las localidades habilitadas, y se registran:

| Métrica | Qué mide | Meta |
|---|---|---|
| `cobertura_nuevos` | % de precios reales de anuncios nuevos dentro del rango | ≈ 80 % |
| `cobertura` | Lo mismo sobre toda la captura de septiembre | ≈ 80 % |
| `ancho_mediano_cop` | Ancho típico del rango en pesos | Lo más angosto posible |
| `razon_sup_inf` | Cuántas veces el límite superior es el inferior | Lo más bajo posible |
| `desvio_meta` | Distancia entre `cobertura_nuevos` y 80 % | 0 |

Un buen rango cubre cerca del 80 % de los precios **sin ser tan ancho que deje de ser útil** para el anfitrión. Las corridas van al experimento `bogoprice-intervalos` de MLflow.

In [1]:
import lightgbm as lgb
import numpy as np
import pandas as pd

from bogoprice import variables_v2
from bogoprice.experimentos import preparar_datos
from bogoprice.metrics import cobertura
from bogoprice.tracking import iniciar_mlflow, registrar_corrida

# Tag de autor de MLflow
AUTOR = "Juan Camilo Piñeros Guzmán"
iniciar_mlflow("bogoprice-intervalos")
datos = preparar_datos()
resultados = []


def intervalo(nombre, version, a_inf, a_sup, registrar=True, **p):
    """Entrena los modelos del percentil inferior y superior, evalúa en septiembre
    y registra la corrida en MLflow. Devuelve las métricas."""
    # Valores base de los dos modelos cuantílicos
    base = {"n_estimators": 1500, "learning_rate": 0.1, "num_leaves": 63,
            "min_child_samples": 30, "subsample": 0.8, "subsample_freq": 1,
            "colsample_bytree": 0.7}
    params = base | p
    cols = variables_v2.columnas(version)

    def modelo(alpha):
        return variables_v2.construir_pipeline(
            lgb.LGBMRegressor(objective="quantile", alpha=alpha, verbose=-1, random_state=0, **params),
            version)

    q_inf = modelo(a_inf).fit(datos.train[cols], datos.y_train)
    q_sup = modelo(a_sup).fit(datos.train[cols], datos.y_train)
    lim_inf = np.exp(q_inf.predict(datos.test[cols]))
    lim_sup = np.exp(q_sup.predict(datos.test[cols]))
    y, nuevo = datos.y_test_cop, datos.es_nuevo

    m = {
        "cobertura": cobertura(y, lim_inf, lim_sup),
        "cobertura_nuevos": cobertura(y[nuevo], lim_inf[nuevo], lim_sup[nuevo]),
        "ancho_mediano_cop": float(np.median(lim_sup - lim_inf)),
        "razon_sup_inf": float(np.median(lim_sup / lim_inf)),
    }
    m["desvio_meta"] = abs(m["cobertura_nuevos"] - 80)
    if registrar:
        registrar_corrida(nombre,
                          {"modelo": "LightGBM cuantílico", "conjunto": version,
                           "alpha_inf": a_inf, "alpha_sup": a_sup, **params,
                           "parametro_variado": p and next(iter(p)) or "percentiles"},
                          m, tags={"autor": AUTOR, "conjunto": version, "modelo": "Intervalo"})
    return m

2026/11/09 02:07:37 INFO mlflow.tracking.fluent: Experiment with name 'bogoprice-intervalos' does not exist. Creating a new experiment.


P99 junio: $1,330,087 | train 17749 | test 17561 | nuevos 1632 | localidades 9 | tipologías 17


## 1. Percentiles del rango

Con los hiperparámetros base, se prueban **4 pares de percentiles**. Mientras más separados, más cobertura pero más ancho el rango.

In [2]:
percentiles = [(0.10, 0.90), (0.075, 0.925), (0.05, 0.95), (0.025, 0.975)]

for version in ["v2", "v2_geo"]:
    for a_inf, a_sup in percentiles:
        etiqueta = f"P{a_inf * 100:g}-P{a_sup * 100:g}"
        nombre = f"intervalo_{etiqueta}_{version}"
        m = intervalo(nombre, version, a_inf, a_sup)
        resultados.append({"corrida": nombre, "conjunto": version, "parametro": "percentiles",
                           "valor": etiqueta, **m})
        print(f"{version:7s} {etiqueta:12s} cobertura nuevos = {m['cobertura_nuevos']:.1f} %  "
              f"ancho = ${m['ancho_mediano_cop']:,.0f}")

🏃 View run intervalo_P10-P90_v2 at: http://localhost:8050/#/experiments/2/runs/0df3cd3c423e4cbab35e6003971b7be7
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2      P10-P90      cobertura nuevos = 66.9 %  ancho = $94,960


🏃 View run intervalo_P7.5-P92.5_v2 at: http://localhost:8050/#/experiments/2/runs/514ae4293b43417a9ef30d465ee1122d
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2      P7.5-P92.5   cobertura nuevos = 71.6 %  ancho = $109,925


🏃 View run intervalo_P5-P95_v2 at: http://localhost:8050/#/experiments/2/runs/42e5cf90445640ad893a166b2946d360
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2      P5-P95       cobertura nuevos = 77.3 %  ancho = $131,166


🏃 View run intervalo_P2.5-P97.5_v2 at: http://localhost:8050/#/experiments/2/runs/aa7d3f960b5d46b48a978ce2b8fae263
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2      P2.5-P97.5   cobertura nuevos = 84.1 %  ancho = $164,032


🏃 View run intervalo_P10-P90_v2_geo at: http://localhost:8050/#/experiments/2/runs/5723eaaa3c2e4cd6b5dd0bf23e4b8053
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2_geo  P10-P90      cobertura nuevos = 61.0 %  ancho = $75,188


🏃 View run intervalo_P7.5-P92.5_v2_geo at: http://localhost:8050/#/experiments/2/runs/387edaecc1844255b43dfbd5d9ce9bcf
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2_geo  P7.5-P92.5   cobertura nuevos = 65.7 %  ancho = $87,250


🏃 View run intervalo_P5-P95_v2_geo at: http://localhost:8050/#/experiments/2/runs/a09f0abd81684cacbcc526519f82cdd0
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2_geo  P5-P95       cobertura nuevos = 74.2 %  ancho = $105,405


🏃 View run intervalo_P2.5-P97.5_v2_geo at: http://localhost:8050/#/experiments/2/runs/87d56a8da34a4fbfa33f58e922e0e08b
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2_geo  P2.5-P97.5   cobertura nuevos = 80.1 %  ancho = $134,515


## 2. Un hiperparámetro a la vez (percentiles P7,5–P92,5)

Con el par de percentiles que apunta al 80 %, se prueban **4 valores de cada hiperparámetro** de los modelos cuantílicos, dejando los demás en la base.

| Hiperparámetro | Base | Valores probados |
|---|---|---|
| `n_estimators` | 600 | 200, 400, 1000, 1500 |
| `learning_rate` | 0,03 | 0,01, 0,02, 0,05, 0,1 |
| `num_leaves` | 31 | 15, 63, 127, 255 |
| `min_child_samples` | 30 | 5, 10, 50, 100 |

In [3]:
A_INF, A_SUP = 0.075, 0.925

pruebas = {
    "n_estimators": [200, 400, 1000, 1500],
    "learning_rate": [0.01, 0.02, 0.05, 0.1],
    "num_leaves": [15, 63, 127, 255],
    "min_child_samples": [5, 10, 50, 100],
}

for version in ["v2", "v2_geo"]:
    for parametro, valores in pruebas.items():
        for valor in valores:
            nombre = f"intervalo_{parametro}{valor}_{version}"
            m = intervalo(nombre, version, A_INF, A_SUP, **{parametro: valor})
            resultados.append({"corrida": nombre, "conjunto": version, "parametro": parametro,
                               "valor": valor, **m})
        print(f"{version:7s} {parametro} listo")

🏃 View run intervalo_n_estimators200_v2 at: http://localhost:8050/#/experiments/2/runs/86865dcfff964f969c5865c2f3551885
🧪 View experiment at: http://localhost:8050/#/experiments/2


🏃 View run intervalo_n_estimators400_v2 at: http://localhost:8050/#/experiments/2/runs/dbc0999acb614153957b1995b4e72c8c
🧪 View experiment at: http://localhost:8050/#/experiments/2


🏃 View run intervalo_n_estimators1000_v2 at: http://localhost:8050/#/experiments/2/runs/2e132ac133b949429b444f3dcedbbd68
🧪 View experiment at: http://localhost:8050/#/experiments/2


🏃 View run intervalo_n_estimators1500_v2 at: http://localhost:8050/#/experiments/2/runs/7111c299043043f39eb9c4f380866583
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2      n_estimators listo


🏃 View run intervalo_learning_rate0.01_v2 at: http://localhost:8050/#/experiments/2/runs/7cf19b170bc9443d97e998da7f55147c
🧪 View experiment at: http://localhost:8050/#/experiments/2


🏃 View run intervalo_learning_rate0.02_v2 at: http://localhost:8050/#/experiments/2/runs/1bb94634134445358fc082eecd819280
🧪 View experiment at: http://localhost:8050/#/experiments/2


... (22 outputs left out)

🏃 View run intervalo_min_child_samples5_v2_geo at: http://localhost:8050/#/experiments/2/runs/10d77b52df11493fbe6e01ddb6ad458d
🧪 View experiment at: http://localhost:8050/#/experiments/2


🏃 View run intervalo_min_child_samples10_v2_geo at: http://localhost:8050/#/experiments/2/runs/871cfcfe99a14134ba9c82694a9d9632
🧪 View experiment at: http://localhost:8050/#/experiments/2


🏃 View run intervalo_min_child_samples50_v2_geo at: http://localhost:8050/#/experiments/2/runs/15acb758083e4653a010de73fb62faf9
🧪 View experiment at: http://localhost:8050/#/experiments/2


🏃 View run intervalo_min_child_samples100_v2_geo at: http://localhost:8050/#/experiments/2/runs/f691786185544870a2293239c4809eb7
🧪 View experiment at: http://localhost:8050/#/experiments/2
v2_geo  min_child_samples listo


## Resumen

Cobertura en anuncios nuevos y ancho del rango para cada prueba.

In [4]:
tabla = pd.DataFrame(resultados)
tabla.pivot_table(index=["parametro", "valor"], columns="conjunto",
                  values=["cobertura_nuevos", "ancho_mediano_cop"]).round(1)

ancho_mediano_cop           cobertura_nuevos  \
conjunto                                    v2    v2_geo               v2   
parametro         valor                                                     
learning_rate     0.01                131915.2  113744.1             79.7   
                  0.02                124566.2  104969.4             77.8   
                  0.05                115748.0   94208.2             74.3   
                  0.1                 109924.8   87249.7             71.6   
min_child_samples 5                   110742.8   86068.8             70.9   
                  10                  107180.0   83931.8             70.4   
                  50                  114296.1   90074.7             73.7   
                  100                 118899.5   97626.8             74.4   
n_estimators      200                 128682.5  109086.4             79.2   
                  400                 120971.9  100803.2             76.2   
                  1000                113526.3   91309.8             72.7   
                  1500                109924.8   87249.7             71.6   
num_leaves        15                  132706.2  111716.4             79.5   
                  63                  109924.8   87249.7             71.6   
                  127                  99336.3   77083.2             67.2   
                  255                  87299.7   68479.6             60.4   
percentiles       P10-P90              94959.9   75188.0             66.9   
                  P2.5-P97.5          164031.9  134514.8             84.1   
                  P5-P95              131166.0  105405.5             77.3   
                  P7.5-P92.5          109924.8   87249.7             71.6   

                                     
conjunto                     v2_geo  
parametro         valor              
learning_rate     0.01         77.8  
                  0.02         74.7  
                  0.05         69.9  
                  0.1          65.7  
min_child_samples 5            65.3  
                  10           64.3  
                  50           68.2  
                  100          70.3  
n_estimators      200          75.6  
                  400          71.4  
                  1000         67.5  
                  1500         65.7  
num_leaves        15           76.5  
                  63           65.7  
                  127          61.6  
                  255          54.2  
percentiles       P10-P90      61.0  
                  P2.5-P97.5   80.1  
                  P5-P95       74.2  
                  P7.5-P92.5   65.7

**Mejores configuraciones:** las más cercanas al 80 % de cobertura en anuncios nuevos y, entre ellas, las de rango más angosto.

In [5]:
columnas = ["conjunto", "parametro", "valor", "cobertura_nuevos", "cobertura",
            "ancho_mediano_cop", "razon_sup_inf", "desvio_meta"]
(tabla[tabla["desvio_meta"] <= 2]
 .sort_values(["conjunto", "ancho_mediano_cop"])
 .groupby("conjunto").head(5)[columnas].round(2))

,conjunto,parametro,valor,cobertura_nuevos,cobertura,ancho_mediano_cop,razon_sup_inf,desvio_meta
8,v2,n_estimators,200,79.23,79.98,128682.53,2.19,0.77
12,v2,learning_rate,0.01,79.66,80.58,131915.21,2.24,0.34
16,v2,num_leaves,15,79.53,80.27,132706.17,2.24,0.47
7,v2_geo,percentiles,P2.5-P97.5,80.09,83.27,134514.75,2.21,0.09
